# Learning Rate Schedules - Exercises

8 exercises covering fixed-step stability, warmup, cosine decay, inverse-square-root schedules, scheduler indexing, cyclical policies, one-cycle training, WSD, and schedule diagnosis.

| Format | Description |
| --- | --- |
| **Problem** | Markdown cell with mathematical context |
| **Your Solution** | Runnable scaffold cell |
| **Solution** | Reference implementation with checks |

### Difficulty Levels

| Level | Exercises | Focus |
| --- | --- | --- |
| * | 1-3 | Core formulas and boundary values |
| ** | 4-6 | Implementation and scheduler mechanics |
| *** | 7-8 | LLM-style WSD and diagnosis |

### Topic Map

| Topic | Exercise |
| --- | --- |
| Stability and warmup-cosine | 1 |
| Transformer inverse-square-root | 2 |
| Step decay and floors | 3 |
| Gradient accumulation indexing | 4 |
| Cyclical schedules | 5 |
| One-cycle policy | 6 |
| Warmup-stable-decay | 7 |
| Training-log diagnosis | 8 |


In [ ]:
import numpy as np
import numpy.linalg as la

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} - {name}")
    if not ok:
        print("  expected:", expected)
        print("  got     :", got)
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} - {name}")
    return cond

# Shared plotting colors when matplotlib is available.
COLORS = {
    "primary": "#0077BB",
    "secondary": "#EE7733",
    "tertiary": "#009988",
    "error": "#CC3311",
    "neutral": "#555555",
}

print("Exercise setup complete.")


---

## Exercise 1 * - Warmup Plus Cosine Schedule

Implement a schedule with linear warmup and cosine decay:

$$
\eta_t =
\begin{cases}
\eta_{\max}t/T_{\mathrm{warm}}, & t<T_{\mathrm{warm}}, \\\\
\eta_{\min}+\frac{1}{2}(\eta_{\max}-\eta_{\min})(1+\cos(\pi u)), & t\ge T_{\mathrm{warm}},
\end{cases}
$$

where

$$
u=\frac{t-T_{\mathrm{warm}}}{T-T_{\mathrm{warm}}}.
$$

Tasks:

(a) Implement the schedule for scalar or array inputs.  
(b) Verify that $\eta_0=0$, $\eta_{T_{\mathrm{warm}}}=\eta_{\max}$, and $\eta_T=\eta_{\min}$.  
(c) Explain why the schedule must clamp $u$ into $[0,1]$ when training continues after the nominal horizon.


In [ ]:
# Exercise 1: Your Solution
def warmup_cosine(step, eta_max, warmup_steps, total_steps, eta_min=0.0):
    # YOUR CODE HERE
    return None

steps = np.array([0, 10, 100])
answer = warmup_cosine(steps, eta_max=0.1, warmup_steps=10, total_steps=100, eta_min=0.001)
print("answer:", answer)


In [ ]:
# Exercise 1: Solution
def warmup_cosine(step, eta_max, warmup_steps, total_steps, eta_min=0.0):
    step = np.asarray(step, dtype=float)
    lr = np.empty_like(step)
    warm = step < warmup_steps
    lr[warm] = eta_max * step[warm] / max(warmup_steps, 1)
    u = (step[~warm] - warmup_steps) / max(total_steps - warmup_steps, 1)
    u = np.clip(u, 0.0, 1.0)
    lr[~warm] = eta_min + 0.5 * (eta_max - eta_min) * (1.0 + np.cos(np.pi * u))
    return lr

header("Exercise 1: Warmup Plus Cosine Schedule")
steps = np.array([0, 10, 55, 100, 120])
values = warmup_cosine(steps, eta_max=0.1, warmup_steps=10, total_steps=100, eta_min=0.001)
print("steps :", steps)
print("values:", values)
check_close("start is zero", values[0], 0.0)
check_close("warmup end reaches peak", values[1], 0.1)
check_close("total step reaches floor", values[3], 0.001)
check_close("after horizon stays at floor", values[4], 0.001)
print("\nTakeaway: warmup-cosine is a piecewise function, so boundary checks are the fastest way to catch scheduler bugs.")


---

## Exercise 2 * - Transformer Inverse-Square-Root Schedule

The original Transformer schedule is

$$
\eta_t=d_{\mathrm{model}}^{-1/2}\min(t^{-1/2},tT_{\mathrm{warm}}^{-3/2}).
$$

Tasks:

(a) Implement the schedule.  
(b) Show that the maximum occurs at $t=T_{\mathrm{warm}}$.  
(c) Compute the peak value for $d_{\mathrm{model}}=512$ and $T_{\mathrm{warm}}=4000$.


In [ ]:
# Exercise 2: Your Solution
def transformer_lr(step, d_model=512, warmup_steps=4000):
    # YOUR CODE HERE
    return None

probe_steps = np.array([1, 4000, 8000])
answer = transformer_lr(probe_steps)
print("answer:", answer)


In [ ]:
# Exercise 2: Solution
def transformer_lr(step, d_model=512, warmup_steps=4000):
    step = np.asarray(step, dtype=float)
    step = np.maximum(step, 1.0)
    return d_model ** -0.5 * np.minimum(step ** -0.5, step * warmup_steps ** -1.5)

header("Exercise 2: Transformer Inverse-Square-Root Schedule")
steps = np.arange(1, 12000)
values = transformer_lr(steps, d_model=512, warmup_steps=4000)
peak_step = steps[np.argmax(values)]
peak_expected = 512 ** -0.5 * 4000 ** -0.5
print(f"peak step: {peak_step}")
print(f"peak value: {values.max():.10f}")
check_close("peak step equals warmup", peak_step, 4000)
check_close("peak value formula", values.max(), peak_expected)
check_true("post-warmup value decays", transformer_lr(np.array([8000]))[0] < values.max())
print("\nTakeaway: the Transformer schedule is linear warmup glued to a slow inverse-square-root tail.")


---

## Exercise 3 * - Multi-Step Decay With a Floor

For milestones $\tau_1,\dots,\tau_m$, define

$$
\eta_t=\max(\eta_{\min},\eta_0\gamma^{\sum_j \mathbf{1}[t\ge \tau_j]}).
$$

Tasks:

(a) Implement the schedule.  
(b) Verify values at milestone boundaries.  
(c) Explain why floors are useful for continued pretraining.


In [ ]:
# Exercise 3: Your Solution
def multi_step_decay(step, eta0, milestones, gamma=0.1, eta_min=0.0):
    # YOUR CODE HERE
    return None

steps = np.array([0, 30, 60, 90])
answer = multi_step_decay(steps, eta0=0.1, milestones=[30, 60], gamma=0.1, eta_min=0.002)
print("answer:", answer)


In [ ]:
# Exercise 3: Solution
def multi_step_decay(step, eta0, milestones, gamma=0.1, eta_min=0.0):
    step = np.asarray(step)
    drops = np.zeros_like(step, dtype=int)
    for tau in milestones:
        drops += (step >= tau)
    return np.maximum(eta_min, eta0 * gamma ** drops)

header("Exercise 3: Multi-Step Decay With a Floor")
steps = np.array([0, 29, 30, 59, 60, 90])
values = multi_step_decay(steps, eta0=0.1, milestones=[30, 60], gamma=0.1, eta_min=0.002)
print("steps :", steps)
print("values:", values)
expected = np.array([0.1, 0.1, 0.01, 0.01, 0.002, 0.002])
check_close("boundary values match", values, expected)
check_true("floor prevents going below eta_min", values[-1] >= 0.002)
print("\nTakeaway: milestone schedules are simple, but every boundary needs an explicit test.")


---

## Exercise 4 ** - Gradient Accumulation Indexing

A training run uses gradient accumulation. The scheduler should advance only when the optimizer updates parameters.

Tasks:

(a) Given $M$ microbatches and accumulation factor $a$, compute optimizer steps.  
(b) If warmup is mistakenly stepped every microbatch, compute the effective warmup measured in optimizer steps.  
(c) Interpret the bug for a transformer run.


In [ ]:
# Exercise 4: Your Solution
def accumulation_summary(microbatches, accumulation, warmup_steps):
    # YOUR CODE HERE
    return None

answer = accumulation_summary(microbatches=8000, accumulation=8, warmup_steps=1000)
print("answer:", answer)


In [ ]:
# Exercise 4: Solution
def accumulation_summary(microbatches, accumulation, warmup_steps):
    optimizer_steps = microbatches // accumulation
    effective_wrong_warmup = warmup_steps / accumulation
    return {
        "optimizer_steps": optimizer_steps,
        "intended_warmup": warmup_steps,
        "wrong_effective_warmup": effective_wrong_warmup,
    }

header("Exercise 4: Gradient Accumulation Indexing")
summary = accumulation_summary(microbatches=8000, accumulation=8, warmup_steps=1000)
print(summary)
check_close("optimizer step count", summary["optimizer_steps"], 1000)
check_close("wrong warmup divided by accumulation", summary["wrong_effective_warmup"], 125)
check_true("bug makes warmup too short", summary["wrong_effective_warmup"] < summary["intended_warmup"])
print("\nTakeaway: scheduler time should usually be optimizer-step time, not microbatch time.")


---

## Exercise 5 ** - Cyclical Learning Rate

Implement a triangular cyclical learning-rate schedule between $\eta_{\min}$ and $\eta_{\max}$.

Tasks:

(a) Implement a triangular cycle with `step_size_up`.  
(b) Add a `triangular2` mode whose amplitude halves each cycle.  
(c) Verify that the second peak in `triangular2` is half the first peak amplitude.


In [ ]:
# Exercise 5: Your Solution
def cyclic_lr(step, base_lr, max_lr, step_size_up, mode="triangular"):
    # YOUR CODE HERE
    return None

steps = np.array([0, 10, 20, 30, 40, 60])
answer = cyclic_lr(steps, base_lr=0.001, max_lr=0.011, step_size_up=20, mode="triangular2")
print("answer:", answer)


In [ ]:
# Exercise 5: Solution
def cyclic_lr(step, base_lr, max_lr, step_size_up, mode="triangular"):
    step = np.asarray(step, dtype=float)
    cycle = np.floor(1 + step / (2 * step_size_up))
    x = np.abs(step / step_size_up - 2 * cycle + 1)
    scale = np.maximum(0.0, 1.0 - x)
    if mode == "triangular2":
        scale = scale / (2 ** (cycle - 1))
    return base_lr + (max_lr - base_lr) * scale

header("Exercise 5: Cyclical Learning Rate")
steps = np.array([0, 20, 40, 60])
tri = cyclic_lr(steps, 0.001, 0.011, 20, mode="triangular")
tri2 = cyclic_lr(steps, 0.001, 0.011, 20, mode="triangular2")
print("steps      :", steps)
print("triangular :", tri)
print("triangular2:", tri2)
first_amp = tri2[1] - 0.001
second_amp = tri2[3] - 0.001
check_close("first peak reaches max", tri2[1], 0.011)
check_close("second peak has half amplitude", second_amp, first_amp / 2)
print("\nTakeaway: cyclic schedules are easiest to verify by checking valleys and peaks.")


---

## Exercise 6 ** - One-Cycle LR and Momentum

A one-cycle schedule increases $\eta_t$ to a maximum, then decays to a very small final value. Momentum often moves inversely.

Tasks:

(a) Implement a two-phase one-cycle schedule.  
(b) Implement inverse momentum scaling.  
(c) Verify that final LR is below initial LR and that momentum is lowest at peak LR.


In [ ]:
# Exercise 6: Your Solution
def one_cycle_lr(step, max_lr, total_steps, pct_start=0.3, div_factor=25, final_div_factor=1e4):
    # YOUR CODE HERE
    return None

def inverse_momentum(lr, lr_min, lr_max, mom_min=0.85, mom_max=0.95):
    # YOUR CODE HERE
    return None

steps = np.array([0, 30, 100])
answer = one_cycle_lr(steps, max_lr=0.01, total_steps=100)
print("answer:", answer)


In [ ]:
# Exercise 6: Solution
def one_cycle_lr(step, max_lr, total_steps, pct_start=0.3, div_factor=25, final_div_factor=1e4):
    step = np.asarray(step, dtype=float)
    initial_lr = max_lr / div_factor
    final_lr = initial_lr / final_div_factor
    up_steps = pct_start * total_steps
    lr = np.empty_like(step)
    up = step <= up_steps
    lr[up] = initial_lr + (max_lr - initial_lr) * step[up] / max(up_steps, 1)
    u = (step[~up] - up_steps) / max(total_steps - up_steps, 1)
    u = np.clip(u, 0.0, 1.0)
    lr[~up] = final_lr + 0.5 * (max_lr - final_lr) * (1.0 + np.cos(np.pi * u))
    return lr

def inverse_momentum(lr, lr_min, lr_max, mom_min=0.85, mom_max=0.95):
    scale = (lr - lr_min) / max(lr_max - lr_min, 1e-12)
    return mom_max - scale * (mom_max - mom_min)

header("Exercise 6: One-Cycle LR and Momentum")
steps = np.arange(0, 101)
lr = one_cycle_lr(steps, max_lr=0.01, total_steps=100, pct_start=0.3)
mom = inverse_momentum(lr, lr.min(), lr.max())
peak_idx = np.argmax(lr)
print(f"initial lr={lr[0]:.8f}, peak lr={lr[peak_idx]:.6f}, final lr={lr[-1]:.10f}")
print(f"initial mom={mom[0]:.4f}, peak-lr mom={mom[peak_idx]:.4f}, final mom={mom[-1]:.4f}")
check_true("final LR below initial LR", lr[-1] < lr[0])
check_true("momentum lowest at peak LR", mom[peak_idx] == mom.min())
check_true("peak occurs near pct_start", abs(peak_idx - 30) <= 1)
print("\nTakeaway: one-cycle uses a high-LR exploration phase and a very low-LR settling phase.")


---

## Exercise 7 *** - Warmup-Stable-Decay Branches

WSD keeps a main branch at high learning rate and spawns final decay branches from selected checkpoints.

Tasks:

(a) Implement a WSD schedule.  
(b) Given branch points, compute the branch final learning rates.  
(c) Verify that each branch reaches $\eta_{\min}$ after its decay horizon.


In [ ]:
# Exercise 7: Your Solution
def wsd_schedule(step, eta_max, warmup_steps, decay_start, total_steps, eta_min):
    # YOUR CODE HERE
    return None

branch_points = [1000, 2000, 3000]
answer = None
print("answer:", answer)


In [ ]:
# Exercise 7: Solution
def wsd_schedule(step, eta_max, warmup_steps, decay_start, total_steps, eta_min):
    step = np.asarray(step, dtype=float)
    lr = np.empty_like(step)
    warm = step < warmup_steps
    stable = (step >= warmup_steps) & (step < decay_start)
    decay = step >= decay_start
    lr[warm] = eta_max * step[warm] / max(warmup_steps, 1)
    lr[stable] = eta_max
    u = (step[decay] - decay_start) / max(total_steps - decay_start, 1)
    u = np.clip(u, 0.0, 1.0)
    lr[decay] = eta_min + 0.5 * (eta_max - eta_min) * (1.0 + np.cos(np.pi * u))
    return lr

header("Exercise 7: Warmup-Stable-Decay Branches")
eta_max = 3e-4
eta_min = 3e-5
warmup = 100
branch_points = [1000, 2000, 3000]
decay_len = 200
final_lrs = []
for start in branch_points:
    local_steps = np.array([start, start + decay_len])
    values = wsd_schedule(local_steps, eta_max, warmup_steps=warmup, decay_start=start, total_steps=start + decay_len, eta_min=eta_min)
    final_lrs.append(values[-1])
    print(f"branch from {start}: start lr={values[0]:.8f}, final lr={values[-1]:.8f}")
check_close("all branches reach eta_min", np.array(final_lrs), np.full(len(branch_points), eta_min))
check_true("stable branch uses peak LR before decay", wsd_schedule(np.array([500]), eta_max, warmup, 1000, 1200, eta_min)[0] == eta_max)
print("\nTakeaway: WSD decouples long high-LR training from short final cooldown branches.")


---

## Exercise 8 *** - Diagnose a Schedule Failure

You are given simplified training logs:

- $\eta_t$ is the learning rate,
- $r_t=\lVert \Delta\boldsymbol{\theta}_t\rVert_2/\lVert \boldsymbol{\theta}_t\rVert_2$ is update ratio,
- $c_t$ is whether clipping occurred.

Tasks:

(a) Diagnose whether the issue is warmup-too-short, peak-LR-too-high, missing decay, or scheduler-resume bug.  
(b) Return a short fix recommendation.  
(c) Test your diagnosis on three cases.


In [ ]:
# Exercise 8: Your Solution
def diagnose_schedule(lr, update_ratio, clipping, train_loss, val_loss):
    # YOUR CODE HERE
    return None

answer = diagnose_schedule(
    lr=np.array([0.001, 0.001, 0.001]),
    update_ratio=np.array([0.01, 0.02, 0.03]),
    clipping=np.array([True, True, True]),
    train_loss=np.array([5.0, 6.0, 8.0]),
    val_loss=np.array([5.2, 6.3, 8.5]),
)
print("answer:", answer)


In [ ]:
# Exercise 8: Solution
def diagnose_schedule(lr, update_ratio, clipping, train_loss, val_loss):
    lr = np.asarray(lr, dtype=float)
    update_ratio = np.asarray(update_ratio, dtype=float)
    clipping = np.asarray(clipping, dtype=bool)
    train_loss = np.asarray(train_loss, dtype=float)
    val_loss = np.asarray(val_loss, dtype=float)

    early_spike = train_loss[min(2, len(train_loss)-1)] > 1.25 * train_loss[0]
    clipping_rate = clipping.mean()
    final_lr_ratio = lr[-1] / max(lr.max(), 1e-12)
    resume_jump = len(lr) > 2 and lr[1] > 2.0 * lr[0] and train_loss[1] > train_loss[0]
    val_gap_worsens = (val_loss[-1] - train_loss[-1]) > (val_loss[0] - train_loss[0])

    if resume_jump:
        return "scheduler-resume bug", "restore scheduler global step and state_dict"
    if early_spike and clipping_rate > 0.5:
        return "warmup-too-short or peak-LR-too-high", "lengthen warmup and lower peak LR"
    if final_lr_ratio > 0.8 and val_gap_worsens:
        return "missing decay", "add cosine/linear cooldown or lower LR floor"
    if update_ratio.max() > 0.02:
        return "peak-LR-too-high", "reduce eta_max or add clipping while retuning"
    return "no obvious schedule failure", "inspect data, optimizer, and model diagnostics"

header("Exercise 8: Diagnose a Schedule Failure")
case_early = diagnose_schedule(
    lr=np.array([0.001, 0.001, 0.001, 0.001]),
    update_ratio=np.array([0.02, 0.04, 0.05, 0.04]),
    clipping=np.array([True, True, True, False]),
    train_loss=np.array([5.0, 6.4, 8.0, 9.0]),
    val_loss=np.array([5.1, 6.8, 8.5, 9.8]),
)
case_resume = diagnose_schedule(
    lr=np.array([0.0001, 0.0010, 0.00098, 0.00096]),
    update_ratio=np.array([0.002, 0.015, 0.014, 0.013]),
    clipping=np.array([False, True, True, False]),
    train_loss=np.array([2.0, 2.8, 2.7, 2.6]),
    val_loss=np.array([2.1, 3.0, 2.9, 2.8]),
)
case_decay = diagnose_schedule(
    lr=np.array([0.001, 0.001, 0.001, 0.001]),
    update_ratio=np.array([0.004, 0.004, 0.004, 0.004]),
    clipping=np.array([False, False, False, False]),
    train_loss=np.array([3.0, 2.0, 1.5, 1.1]),
    val_loss=np.array([3.1, 2.4, 2.3, 2.4]),
)
print("early case :", case_early)
print("resume case:", case_resume)
print("decay case :", case_decay)
check_true("early case identifies warmup or peak LR", "warmup" in case_early[0] or "peak" in case_early[0])
check_true("resume case identifies resume bug", case_resume[0] == "scheduler-resume bug")
check_true("decay case identifies missing decay", case_decay[0] == "missing decay")
print("\nTakeaway: logs should include LR, update ratio, clipping, train loss, and validation loss together.")


---

## What to Review After Finishing

- [ ] Can you write warmup-cosine from memory?
- [ ] Can you explain why the Transformer schedule peaks at $T_{\mathrm{warm}}$?
- [ ] Can you tell optimizer steps from microbatch steps?
- [ ] Can you identify when a schedule resume state is wrong?
- [ ] Can you choose between cosine and WSD based on compute-budget certainty?

## References

1. Vaswani et al. (2017), _Attention Is All You Need_.
2. Loshchilov and Hutter (2017), _SGDR: Stochastic Gradient Descent with Warm Restarts_.
3. Smith (2015), _Cyclical Learning Rates for Training Neural Networks_.
4. Smith and Topin (2017), _Super-Convergence_.
5. Hu et al. (2024), _Understanding Warmup-Stable-Decay Learning Rates_.
